# Сбор 320 фотографий бабочек из iNaturalist

В этом ноутбуке я собираю фотографии взрослых бабочек для дальнейшей разметки рамками. Сначала получаю кандидатов с лицензией CC0, затем вручную отбираю 320 изображений.

| Категория | Количество |
|---|---:|
| `single_clear` — хорошо видимая бабочка | 150 |
| `small` — маленькая бабочка | 70 |
| `multiple` — несколько бабочек | 13 |
| `occluded` — частично закрытая бабочка | 50 |
| `complex_background` — сложный фон | 37 |
| **Всего** | **320** |

После просмотра 623 кандидатов я нашла только 13 подходящих фотографий с несколькими бабочками, поэтому остальные места распределила между другими категориями. Ещё 80 фотографий без бабочек соберу отдельно.


## 0. Подготовка

Устанавливаю библиотеки, необходимые для сбора, просмотра и проверки фотографий.

In [1]:
%pip install -q requests pandas pillow ipywidgets ImageHash


[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Настройки

Задаю категории и нужное количество фотографий, а также папки для хранения результатов.

In [2]:
from pathlib import Path
from datetime import datetime, timezone
from html import escape
from urllib.parse import urlparse
from itertools import combinations
import json
import time
import hashlib
import shutil

import pandas as pd
import requests
import ipywidgets as widgets
from PIL import Image
from IPython.display import display
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

PROJECT_DIR = Path.home() / "butterfly-detection"
RUN_DIR = PROJECT_DIR / "positive_320_from_scratch"
DATA_DIR = RUN_DIR / "data"
IMAGES_DIR = RUN_DIR / "images"

DATA_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

CANDIDATES_PATH = DATA_DIR / "candidates_positive.csv"
SELECTIONS_PATH = DATA_DIR / "selections_positive.csv"
FETCH_STATE_PATH = DATA_DIR / "positive_fetch_state.json"
VALIDATION_PATH = DATA_DIR / "positive_validation.csv"
METADATA_PATH = DATA_DIR / "metadata_positive.csv"
BBOX_INPUT_PATH = DATA_DIR / "bbox_input_positive.csv"

API_URL = "https://api.inaturalist.org/v1/observations"
TAXON_ID = 47224
PER_PAGE = 200
TARGET_CANDIDATES = 800
QUOTAS = {
    "single_clear": 150,
    "small": 70,
    "multiple": 13,
    "occluded": 50,
    "complex_background": 37,
}
TARGET_SELECTED = sum(QUOTAS.values())

CANDIDATE_COLUMNS = [
    "photo_id", "observation_id", "observation_url", "photo_url",
    "original_url", "license_code", "attribution",
]
SELECTION_COLUMNS = ["photo_id", "category", "reviewed_at"]

retry = Retry(
    total=4,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
)
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=retry))
session.headers.update({"User-Agent": "CourseworkButterflyDataset/1.0 (academic data collection)"})


def save_csv(df_to_save, path):
    """Безопасная перезапись CSV: сначала временный файл, затем замена."""
    temporary_path = path.with_name(path.name + ".tmp")
    df_to_save.to_csv(temporary_path, index=False)
    temporary_path.replace(path)


def load_candidates():
    if not CANDIDATES_PATH.exists():
        return pd.DataFrame(columns=CANDIDATE_COLUMNS)
    result = pd.read_csv(CANDIDATES_PATH)
    for col in CANDIDATE_COLUMNS:
        if col not in result:
            result[col] = pd.NA
    result["photo_id"] = pd.to_numeric(result["photo_id"], errors="coerce")
    result = result.dropna(subset=["photo_id"]).copy()
    result["photo_id"] = result["photo_id"].astype("int64")
    return result[CANDIDATE_COLUMNS].drop_duplicates("photo_id").reset_index(drop=True)


def load_selections():
    if not SELECTIONS_PATH.exists():
        return pd.DataFrame(columns=SELECTION_COLUMNS)

    result = pd.read_csv(SELECTIONS_PATH).fillna("")

    legacy_columns = {"small_object", "multiple_objects", "occluded", "complex_background", "notes"}
    if legacy_columns.intersection(result.columns):
        backup = SELECTIONS_PATH.with_name("selections_positive_before_simplification.csv")
        if not backup.exists():
            shutil.copy2(SELECTIONS_PATH, backup)
            print("Сохранена резервная копия прежнего отбора:", backup)

    for col in SELECTION_COLUMNS:
        if col not in result.columns:
            result[col] = ""
    result["photo_id"] = pd.to_numeric(result["photo_id"], errors="coerce")
    result = result.dropna(subset=["photo_id"]).copy()
    result["photo_id"] = result["photo_id"].astype("int64")
    return result[SELECTION_COLUMNS].drop_duplicates("photo_id", keep="last")


def original_url(url):
    for size_name in ("square", "thumb", "small", "medium", "large"):
        url = url.replace(f"/{size_name}.", "/original.")
    return url


print("Основная папка проекта:", PROJECT_DIR)
print("Новая рабочая папка:", RUN_DIR)
print("План: отобрать", TARGET_SELECTED, "фотографий из", TARGET_CANDIDATES, "или более кандидатов")

Основная папка проекта: /Users/ekaterinaseloceva/butterfly-detection
Новая рабочая папка: /Users/ekaterinaseloceva/butterfly-detection/positive_320_from_scratch
План: отобрать 320 фотографий из 800 или более кандидатов


## 2. Сохранённые данные

Проверяю, сколько фотографий-кандидатов и результатов отбора уже сохранено. Это позволяет продолжать работу после перезапуска.

In [3]:
print("Новая рабочая папка:", RUN_DIR)
print("Кандидатов в этом наборе:", len(load_candidates()))
print("Принятых решений в этом наборе:", len(load_selections()))


Новая рабочая папка: /Users/ekaterinaseloceva/butterfly-detection/positive_320_from_scratch
Кандидатов в этом наборе: 963
Принятых решений в этом наборе: 626


## 3. Получение фотографий-кандидатов

Через API iNaturalist получаю не менее 800 кандидатов с лицензией CC0. Пока сохраняю только ссылки и информацию о фотографиях, без самих изображений. Повторяющиеся фотографии исключаю, а новые результаты сохраняю, чтобы не потерять прогресс.

In [4]:
def fetch_candidates(min_total=TARGET_CANDIDATES, max_pages_this_run=25):
    candidates = load_candidates()
    existing_ids = set(candidates["photo_id"].tolist())

    if FETCH_STATE_PATH.exists():
        state = json.loads(FETCH_STATE_PATH.read_text(encoding="utf-8"))
    else:
        state = {"next_page": 1}

    page = int(state.get("next_page", 1))
    fetched_pages = 0

    while len(existing_ids) < min_total and fetched_pages < max_pages_this_run:
        params = {
            "taxon_id": TAXON_ID,
            "photos": "true",
            "photo_license": "cc0",
            "per_page": PER_PAGE,
            "page": page,
        }
        response = session.get(API_URL, params=params, timeout=30)
        response.raise_for_status()
        observations = response.json().get("results", [])

        if not observations:
            print("API больше не вернул наблюдений. Остановлено на странице", page)
            break

        new_rows = []
        for obs in observations:
            observation_id = obs.get("id")
            if observation_id is None:
                continue
            for photo in obs.get("photos") or []:
                if str(photo.get("license_code") or "").lower() != "cc0":
                    continue
                photo_id, url = photo.get("id"), photo.get("url")
                if photo_id is None or not isinstance(url, str) or not url.startswith("https://"):
                    continue
                photo_id = int(photo_id)
                if photo_id in existing_ids:
                    continue
                existing_ids.add(photo_id)
                new_rows.append({
                    "photo_id": photo_id,
                    "observation_id": int(observation_id),
                    "observation_url": f"https://www.inaturalist.org/observations/{observation_id}",
                    "photo_url": url,
                    "original_url": original_url(url),
                    "license_code": "cc0",
                    "attribution": photo.get("attribution") or "",
                })

        if new_rows:
            candidates = pd.concat([candidates, pd.DataFrame(new_rows)], ignore_index=True)
            candidates = candidates.drop_duplicates("photo_id").reset_index(drop=True)
            save_csv(candidates, CANDIDATES_PATH)

        page += 1
        fetched_pages += 1
        FETCH_STATE_PATH.write_text(
            json.dumps({"next_page": page}, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        print(f"Страница {page - 1}: +{len(new_rows)} фото; всего {len(candidates)} кандидатов")
        time.sleep(1.0)

    print(f"Готово: {len(candidates)} уникальных кандидатов")
    if len(candidates) < min_total:
        print("Цель по кандидатам ещё не достигнута. Можно запустить ячейку ещё раз.")
    return candidates


df = fetch_candidates(min_total=TARGET_CANDIDATES)

Готово: 963 уникальных кандидатов


## 4. Проверка кандидатов

Проверяю количество фотографий, уникальность их ID, ссылки и лицензию CC0. Подходящие изображения выберу позже при просмотре.

In [5]:
df = load_candidates()
print("Кандидатов:", len(df))
print("Уникальных photo_id:", df["photo_id"].nunique())
print("Лицензии:", df["license_code"].value_counts(dropna=False).to_dict())
print("Наблюдений:", df["observation_id"].nunique())
assert df["photo_id"].is_unique, "Есть повторяющиеся photo_id"
assert df["license_code"].str.lower().eq("cc0").all(), "Есть фотографии не с CC0"
assert df["original_url"].fillna("").str.startswith("https://").all()
display(df.head(5))

Кандидатов: 963
Уникальных photo_id: 963
Лицензии: {'cc0': 963}
Наблюдений: 600


,photo_id,observation_id,observation_url,photo_url,original_url,license_code,attribution
0,762723421,414327089,https://www.inaturalist.org/observations/41432...,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,cc0,no rights reserved
1,762723438,414327089,https://www.inaturalist.org/observations/41432...,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,cc0,no rights reserved
2,762723449,414327089,https://www.inaturalist.org/observations/41432...,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,cc0,no rights reserved
3,762720003,414325393,https://www.inaturalist.org/observations/41432...,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,cc0,no rights reserved
4,762719515,414325087,https://www.inaturalist.org/observations/41432...,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,cc0,no rights reserved


## 5. Ручной отбор фотографий

Просматриваю кандидатов и выбираю для каждой фотографии одну категорию. Неподходящие отмечаю как `reject`, сомнительные - как `unsure`. Программа сохраняет выбор и показывает, сколько фотографий осталось набрать.

In [6]:
df = load_candidates()
selection_df = load_selections()
selections = {int(r["photo_id"]): r.to_dict() for _, r in selection_df.iterrows()}

category_options = [
    ("Не выбрано", ""),
    ("Одна хорошо видимая бабочка", "single_clear"),
    ("Маленькая бабочка", "small"),
    ("Несколько бабочек", "multiple"),
    ("Перекрытая / обрезанная", "occluded"),
    ("Сложный фон", "complex_background"),
    ("Отклонить", "reject"),
    ("Нужно проверить", "unsure"),
]

index_state = {"i": 0}
category_dropdown = widgets.Dropdown(
    options=category_options, description="Категория:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="480px"),
)
info = widgets.HTML()
photo_html = widgets.HTML()
progress_html = widgets.HTML()
status = widgets.HTML()
prev_button = widgets.Button(description="← Назад")
save_button = widgets.Button(description="Сохранить и далее", button_style="success")
next_button = widgets.Button(description="Далее →")
unreviewed_button = widgets.Button(description="К нерассмотренной", button_style="info")


def category_counts():
    return {
        category: sum(item.get("category") == category for item in selections.values())
        for category in QUOTAS
    }


def photo_preview_url(url):
    return str(url).replace("/square.", "/medium.").replace("/original.", "/medium.")


def show_photo():
    if df.empty:
        status.value = "Нет кандидатов. Сначала выполни загрузку из API."
        return
    row = df.iloc[index_state["i"]]
    pid = int(row["photo_id"])
    previous = selections.get(pid, {})
    category_dropdown.value = previous.get("category", "") or ""
    c = category_counts()
    total_selected = sum(c.values())
    info.value = (
        f"<b>Фото {index_state['i'] + 1} из {len(df)}</b> | "
        f"photo_id: <b>{pid}</b> | выбрано: <b>{total_selected}/{TARGET_SELECTED}</b>"
    )
    progress_html.value = " | ".join(
        f"{escape(name)}: {count}/{QUOTAS[name]}" for name, count in c.items()
    )
    obs_id = row["observation_id"]
    other_selected = [
        int(r["photo_id"]) for _, r in df.iterrows()
        if r["observation_id"] == obs_id
        and int(r["photo_id"]) != pid
        and selections.get(int(r["photo_id"]), {}).get("category") in QUOTAS
    ]
    warning = (
        f"<p style='color:#a66d00'>Из этого же наблюдения уже выбрано: "
        f"{', '.join(map(str, other_selected))}. Проверь сходство кадров.</p>"
        if other_selected else ""
    )
    url = escape(photo_preview_url(row["photo_url"]), quote=True)
    obs_url = escape(str(row["observation_url"]), quote=True)
    full_url = escape(str(row["original_url"]), quote=True)
    photo_html.value = (
        f'<div><img src="{url}" style="max-width:700px;max-height:550px;object-fit:contain" '
        f'alt="photo {pid}"></div>'
        f'<p><a href="{full_url}" target="_blank">Открыть оригинал</a> | '
        f'<a href="{obs_url}" target="_blank">Наблюдение на iNaturalist</a></p>{warning}'
    )


def persist_selections():
    saved = pd.DataFrame(list(selections.values()), columns=SELECTION_COLUMNS)
    save_csv(saved, SELECTIONS_PATH)


def go_next_unreviewed(_=None):
    if df.empty:
        return
    n = len(df)
    current = index_state["i"]
    for step in range(1, n + 1):
        j = (current + step) % n
        if int(df.iloc[j]["photo_id"]) not in selections:
            index_state["i"] = j
            status.value = ""
            show_photo()
            return
    status.value = "Все кандидаты просмотрены. Для продолжения добавь новые через API."


def save_selection(_):
    if df.empty:
        return
    row = df.iloc[index_state["i"]]
    pid = int(row["photo_id"])
    new_category = category_dropdown.value
    if not new_category:
        status.value = "<b>Сначала выбери категорию.</b>"
        return
    old_category = selections.get(pid, {}).get("category")
    if new_category in QUOTAS and new_category != old_category:
        if category_counts()[new_category] >= QUOTAS[new_category]:
            status.value = f"<b>Квота {new_category} уже заполнена — выбери другое решение.</b>"
            return
    selections[pid] = {
        "photo_id": pid,
        "category": new_category,
        "reviewed_at": datetime.now(timezone.utc).isoformat(),
    }
    persist_selections()
    status.value = f"Сохранено: {pid} → {escape(new_category)}"
    if index_state["i"] < len(df) - 1:
        index_state["i"] += 1
    show_photo()


def prev_photo(_):
    index_state["i"] = max(0, index_state["i"] - 1)
    show_photo()


def next_photo(_):
    index_state["i"] = min(len(df) - 1, index_state["i"] + 1)
    show_photo()


prev_button.on_click(prev_photo)
next_button.on_click(next_photo)
save_button.on_click(save_selection)
unreviewed_button.on_click(go_next_unreviewed)

ui = widgets.VBox([
    info,
    photo_html,
    category_dropdown,
    widgets.HBox([prev_button, save_button, next_button, unreviewed_button]),
    progress_html,
    status,
])
display(ui)

unreviewed_positions = [i for i, pid in enumerate(df["photo_id"])
                        if int(pid) not in selections]
if unreviewed_positions:
    index_state["i"] = unreviewed_positions[0]
show_photo()

## 6. Прогресс отбора

Смотрю, сколько фотографий уже выбрано в каждой категории, сколько осталось, а также сколько изображений отклонено или ещё не просмотрено.

In [7]:
selection_df = load_selections()
counts = selection_df["category"].value_counts()
progress = pd.DataFrame({"category": list(QUOTAS), "target": list(QUOTAS.values())})
progress["selected"] = progress["category"].map(counts).fillna(0).astype(int)
progress["remaining"] = progress["target"] - progress["selected"]
display(progress)
print("Выбрано:", progress["selected"].sum(), "/", TARGET_SELECTED)
print("Отклонено:", int(counts.get("reject", 0)), "| Неоднозначно:", int(counts.get("unsure", 0)))
print("Не просмотрено:", len(load_candidates()) - len(selection_df))

,category,target,selected,remaining
0,single_clear,150,150,0
1,small,70,70,0
2,multiple,13,13,0
3,occluded,50,50,0
4,complex_background,37,37,0


Выбрано: 320 / 320
Отклонено: 306 | Неоднозначно: 0
Не просмотрено: 337


## 7. Дополнительные кандидаты

Если подходящих фотографий не хватит, могу загрузить ещё одну партию. Уже собранные фотографии и результаты отбора при этом сохранятся. По умолчанию дополнительная загрузка выключена.

In [8]:
ADD_MORE = False
if ADD_MORE:
    current_count = len(load_candidates())
    df = fetch_candidates(min_total=current_count + 400)
else:
    print("Новая партия не запрошена. Чтобы добавить 400 кандидатов, поставь ADD_MORE = True.")

Новая партия не запрошена. Чтобы добавить 400 кандидатов, поставь ADD_MORE = True.


## 8. Список отобранных фотографий

Когда заполню все квоты, объединяю выбранные фотографии с их ссылками и сохраняю общий список из 320 изображений.

In [9]:
selection_df = load_selections()
selected = selection_df[selection_df["category"].isin(QUOTAS)].copy()
category_counts_final = selected["category"].value_counts().to_dict()
complete = all(category_counts_final.get(k, 0) == v for k, v in QUOTAS.items())

manifest = selected.merge(load_candidates(), on="photo_id", how="left", validate="one_to_one")
missing_sources = manifest["original_url"].isna().sum()
print("Выбрано:", len(manifest), "/", TARGET_SELECTED)
print("По категориям:", category_counts_final)
print("Нет ссылок на источник:", missing_sources)

if complete and missing_sources == 0:
    save_csv(manifest, METADATA_PATH)
    print("Предварительный манифест:", METADATA_PATH)
else:
    print("Пока рано создавать финальный датасет: заполни все квоты и проверь источники.")

Выбрано: 320 / 320
По категориям: {'single_clear': 150, 'small': 70, 'occluded': 50, 'complex_background': 37, 'multiple': 13}
Нет ссылок на источник: 0
Предварительный манифест: /Users/ekaterinaseloceva/butterfly-detection/positive_320_from_scratch/data/metadata_positive.csv


## 9. Скачивание фотографий

Скачиваю оригиналы выбранных 320 фотографий и проверяю, что они открываются. Уже скачанные файлы использую повторно, а ошибки сохраняю для дальнейшей проверки.

In [10]:
import io


def download_originals():
    if not METADATA_PATH.exists():
        print("Нет готового metadata_positive.csv — сначала заполни квоты.")
        return

    manifest = pd.read_csv(METADATA_PATH)
    if len(manifest) != TARGET_SELECTED:
        print("Не 320 фотографий: скачивание остановлено.")
        return

    previous = {}
    if VALIDATION_PATH.exists():
        history = pd.read_csv(VALIDATION_PATH)
        previous = {int(r["photo_id"]): r.to_dict() for _, r in history.iterrows()}

    results = previous.copy()
    for i, row in manifest.iterrows():
        pid = int(row["photo_id"])
        url = str(row["original_url"])
        ext = Path(urlparse(url).path).suffix.lower()
        if ext not in (".jpg", ".jpeg", ".png", ".webp"):
            ext = ".jpg"
        path = IMAGES_DIR / f"{pid}{ext}"

        try:
            if not path.exists() or path.stat().st_size == 0:
                response = session.get(url, timeout=60, stream=True)
                response.raise_for_status()
                temp = path.with_name(path.name + ".part")
                try:
                    with open(temp, "wb") as f:
                        for chunk in response.iter_content(chunk_size=1024 * 1024):
                            if chunk:
                                f.write(chunk)
                    with Image.open(temp) as im:
                        im.verify()
                    temp.replace(path)
                finally:
                    response.close()
                    if temp.exists():
                        temp.unlink()

            with Image.open(path) as im:
                im.load()
                width, height, image_format = im.width, im.height, im.format
            digest = hashlib.sha256(path.read_bytes()).hexdigest()
            results[pid] = {
                "photo_id": pid,
                "filename": path.name,
                "width": width, "height": height,
                "format": image_format,
                "size_mb": round(path.stat().st_size / (1024 ** 2), 2),
                "sha256": digest,
                "valid": True, "error": "",
            }
        except Exception as exc:
            results[pid] = {
                "photo_id": pid,
                "filename": path.name,
                "width": None, "height": None, "format": "",
                "size_mb": None, "sha256": "",
                "valid": False, "error": str(exc),
            }

        save_csv(pd.DataFrame(list(results.values())), VALIDATION_PATH)
        if (i + 1) % 10 == 0 or (i + 1) == len(manifest):
            processed_ids = manifest.iloc[:i + 1]["photo_id"].astype(int)
            successful = sum(
                str(results[pid]["valid"]).lower() == "true"
                for pid in processed_ids
            )
            print(f"Обработано: {i+1}/{len(manifest)}; корректных файлов: {successful}")
        time.sleep(0.1)

    print("Результаты проверки и загрузки:", VALIDATION_PATH)


DOWNLOAD_ORIGINALS = True
if DOWNLOAD_ORIGINALS:
    download_originals()
else:
    print("Скачивание выключено. После отбора 320 фото поставь DOWNLOAD_ORIGINALS = True.")

Обработано: 10/320; корректных файлов: 10
Обработано: 20/320; корректных файлов: 20
Обработано: 30/320; корректных файлов: 30
Обработано: 40/320; корректных файлов: 40
Обработано: 50/320; корректных файлов: 50
Обработано: 60/320; корректных файлов: 60
Обработано: 70/320; корректных файлов: 70
Обработано: 80/320; корректных файлов: 80
Обработано: 90/320; корректных файлов: 90
Обработано: 100/320; корректных файлов: 100
Обработано: 110/320; корректных файлов: 110
Обработано: 120/320; корректных файлов: 120
Обработано: 130/320; корректных файлов: 130
Обработано: 140/320; корректных файлов: 140
Обработано: 150/320; корректных файлов: 150
Обработано: 160/320; корректных файлов: 160
Обработано: 170/320; корректных файлов: 170
Обработано: 180/320; корректных файлов: 180
Обработано: 190/320; корректных файлов: 190
Обработано: 200/320; корректных файлов: 200
Обработано: 210/320; корректных файлов: 210
Обработано: 220/320; корректных файлов: 220
Обработано: 230/320; корректных файлов: 230
Обрабо

## 10. Проверка похожих фотографий

Ищу одинаковые и очень похожие изображения, в том числе снимки из одного наблюдения.

In [11]:
import imagehash

if not VALIDATION_PATH.exists():
    print("Сначала скачай отобранные оригиналы (шаг 9).")
else:
    checks = pd.read_csv(VALIDATION_PATH)
    good = checks[checks["valid"].astype(str).str.lower() == "true"].copy()
    repeated_sha = good[good.duplicated("sha256", keep=False)].sort_values("sha256")
    print("Повторяющиеся файлы (SHA-256):", len(repeated_sha))
    display(repeated_sha[["photo_id", "filename", "sha256"]])

    perceptual_hashes = {}
    for _, item in good.iterrows():
        try:
            with Image.open(IMAGES_DIR / str(item["filename"])) as im:
                perceptual_hashes[int(item["photo_id"])] = imagehash.phash(im.convert("RGB"))
        except Exception as exc:
            print("Не удалось вычислить pHash для", item["photo_id"], ":", exc)

    similar_pairs = [
        {"photo_id_1": a, "photo_id_2": b, "phash_distance": perceptual_hashes[a] - perceptual_hashes[b]}
        for a, b in combinations(perceptual_hashes, 2)
        if perceptual_hashes[a] - perceptual_hashes[b] <= 6
    ]
    print("Подозрительно похожих пар (pHash <= 6):", len(similar_pairs))
    display(pd.DataFrame(similar_pairs))

    if METADATA_PATH.exists():
        manifest = pd.read_csv(METADATA_PATH)
        same_observation = manifest[
            manifest.duplicated("observation_id", keep=False)
        ].sort_values("observation_id")
        print("Фото из наблюдений с несколькими выбранными кадрами:", len(same_observation))
        display(same_observation[["photo_id", "observation_id", "category"]])

Повторяющиеся файлы (SHA-256): 0


,photo_id,filename,sha256


Подозрительно похожих пар (pHash <= 6): 0


""


Фото из наблюдений с несколькими выбранными кадрами: 100


,photo_id,observation_id,category
313,761312847,413626831,complex_background
312,761312810,413626831,occluded
261,761459946,413699016,occluded
262,761460577,413699016,small
257,761463940,413700794,small
...,...,...,...
11,762682720,414307280,single_clear
10,762683658,414307722,complex_background
9,762683666,414307722,single_clear
5,762688898,414310509,small


## 11. Подготовка данных для bbox

Проверяю, что все 320 фотографий уникальны, имеют лицензию CC0 и успешно скачаны. Если всё в порядке, сохраняю для разметки только ID и ссылки, без предварительных категорий.

In [12]:
if not (METADATA_PATH.exists() and VALIDATION_PATH.exists()):
    print("Нет итогового манифеста и/или проверки оригиналов. Экспорт пока не выполнен.")
else:
    manifest = pd.read_csv(METADATA_PATH)
    checks = pd.read_csv(VALIDATION_PATH)
    merged = manifest.merge(checks, on="photo_id", how="left", validate="one_to_one")
    counts = manifest["category"].value_counts().to_dict()
    exact_quotas = all(counts.get(k, 0) == v for k, v in QUOTAS.items())
    all_valid = merged["valid"].astype(str).str.lower().eq("true").all()
    all_unique = manifest["photo_id"].is_unique
    all_cc0 = manifest["license_code"].str.lower().eq("cc0").all()
    files_exist = all(
        (IMAGES_DIR / str(name)).exists() for name in merged["filename"].dropna()
    ) and merged["filename"].notna().all()

    print("Квоты:", exact_quotas)
    print("Оригиналы проверены:", all_valid)
    print("Уникальные ID:", all_unique)
    print("Все CC0:", all_cc0)
    print("Файлы на диске:", files_exist)

    if exact_quotas and all_valid and all_unique and all_cc0 and files_exist:
        bbox_input = manifest[["photo_id", "original_url"]].rename(
            columns={"original_url": "image_url"}
        )
        save_csv(bbox_input, BBOX_INPUT_PATH)
        print("Готовый список для bbox:", BBOX_INPUT_PATH)
        print("Фотографий:", len(bbox_input))
    else:
        print("Экспорт пока остановлен. Исправь проблемы, затем запусти проверку снова.")

Квоты: True
Оригиналы проверены: True
Уникальные ID: True
Все CC0: True
Файлы на диске: True
Готовый список для bbox: /Users/ekaterinaseloceva/butterfly-detection/positive_320_from_scratch/data/bbox_input_positive.csv
Фотографий: 320


## Результат

После отбора у меня будут 320 проверенных фотографий, таблица с их данными и отдельный список ссылок для исполнителей bbox. Файлы и результаты отбора сохраняются в папке проекта. Ещё 80 фотографий без бабочек соберу отдельно.